# Lab 06: Scan Matching Test

**12062 Algorithmic Robotics | Week 8**

This notebook tests your `scan_matcher.py` implementation. Copy your completed `scan_matcher.py` into the same folder as this notebook, then run each cell in order.

Three test environments are provided, each with a reference scan and a shifted scan (ground truth known). Change the environment name in Cells 3 and 4 to test all three.

## Cell 1 — Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scan_matcher import ScanMatcher

#Import the scan matcher code you just wrote. If this line causes an error, make sure your scan_matcher.py is in the same directory as this file.

print('Import successful — your scan_matcher.py was found.')

## Cell 2 — Generate Test Environments

Three environments, each with a reference scan (`scan_ref`) and a new scan (`scan_new`) shifted by a known ground truth pose. The odometry initial guess has deliberate error added so you can see the difference before and after matching.

In [ ]:
# --- Helper: transform scan by a relative pose ---
def transform_scan(points, dx, dy, dtheta):
    c, s = np.cos(dtheta), np.sin(dtheta)
    rx = c * points[:, 0] - s * points[:, 1] + dx
    ry = s * points[:, 0] + c * points[:, 1] + dy
    return np.column_stack([rx, ry])


# --- Environment 1: Corner (L-shaped, feature-rich) ---
def make_corner(d=3.0, length=4.0, n=60):
    y1 = np.linspace(-length / 2, d, n)
    x1 = np.full_like(y1, d)
    x2 = np.linspace(-length / 2, d, n)
    y2 = np.full_like(x2, d)
    return np.vstack([np.column_stack([x1, y1]), np.column_stack([x2, y2])])


# --- Environment 2: Wall (single flat wall) ---
def make_wall(d=3.0, length=6.0, n=100):
    y = np.linspace(-length / 2, length / 2, n)
    x = np.full_like(y, d)
    return np.column_stack([x, y])


# --- Environment 3: Corridor (two parallel walls — degenerate) ---
def make_corridor(width=2.0, length=8.0, n=80):
    y = np.linspace(-length / 2, length / 2, n)
    left = np.column_stack([np.full_like(y, -width / 2), y])
    right = np.column_stack([np.full_like(y, width / 2), y])
    return np.vstack([left, right])


# Ground truth relative pose (same for all environments)
TRUE_POSE = np.array([0.12, -0.10, 0.08])

# Odometry initial guess (deliberately noisy)
ODOM_GUESS = TRUE_POSE + np.array([0.06, -0.05, 0.03])

# Build scan pairs
envs = {}
for name, make_fn in [('corner', make_corner), ('wall', make_wall), ('corridor', make_corridor)]:
    ref = make_fn()
    new = transform_scan(ref, -TRUE_POSE[0], -TRUE_POSE[1], -TRUE_POSE[2])
    envs[name] = {'scan_ref': ref, 'scan_new': new}

# Visualise all three
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, data) in zip(axes, envs.items()):
    ax.scatter(data['scan_ref'][:, 0], data['scan_ref'][:, 1], s=5, c='blue', label='scan_ref')
    ax.scatter(data['scan_new'][:, 0], data['scan_new'][:, 1], s=5, c='red', alpha=0.5, label='scan_new')
    ax.plot(0, 0, 'k^', markersize=8)
    ax.set_title(name); ax.set_aspect('equal'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.suptitle('Test Environments — Reference (blue) and Shifted (red) Scans', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Ground truth pose: dx={TRUE_POSE[0]:.2f} m, dy={TRUE_POSE[1]:.2f} m, dtheta={TRUE_POSE[2]:.2f} rad')
print(f'Odometry guess:    dx={ODOM_GUESS[0]:.2f} m, dy={ODOM_GUESS[1]:.2f} m, dtheta={ODOM_GUESS[2]:.2f} rad')

## Cell 3 — Run Scan Matching and Plot Alignment

Change `ENV` below to `'corner'`, `'wall'`, or `'corridor'` to test different environments. Run the cell to see before/after alignment.

**Portfolio artifact: save the figure for each environment.**

In [ ]:
# ============================================================
# CHANGE THIS to 'corner', 'wall', or 'corridor'
ENV = 'corner'
# ============================================================

scan_ref = envs[ENV]['scan_ref']
scan_new = envs[ENV]['scan_new']

# Create matcher with params.yaml values
matcher = ScanMatcher(
    search_x=0.15, search_y=0.15, search_theta=0.1,
    resolution_x=0.03, resolution_y=0.03, resolution_theta=0.02,
    local_grid_size=500, local_grid_resolution=0.05, min_score=0.3
)

# Run scan matching
matched_pose, covariance, score = matcher.match(scan_ref, scan_new, ODOM_GUESS)
match_error = matched_pose - TRUE_POSE
odom_error = ODOM_GUESS - TRUE_POSE

# --- Plot before / after alignment ---
def overlay(ax, ref, new, pose, title):
    c, s = np.cos(pose[2]), np.sin(pose[2])
    ax.scatter(ref[:, 0], ref[:, 1], s=8, c='blue', alpha=0.6, label='scan_ref')
    ax.scatter(c * new[:, 0] - s * new[:, 1] + pose[0],
               s * new[:, 0] + c * new[:, 1] + pose[1],
               s=8, c='red', alpha=0.6, label='scan_new (transformed)')
    ax.set_title(title); ax.set_aspect('equal'); ax.legend(fontsize=9)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
overlay(axes[0], scan_ref, scan_new, ODOM_GUESS,
        f'BEFORE — Odometry Guess\nerror: {np.linalg.norm(odom_error[:2]):.4f} m, {np.degrees(odom_error[2]):.1f} deg')
overlay(axes[1], scan_ref, scan_new, matched_pose,
        f'AFTER — Scan Match Result\nerror: {np.linalg.norm(match_error[:2]):.4f} m, {np.degrees(match_error[2]):.1f} deg')
plt.suptitle(f'Scan Alignment — {ENV}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'scan_alignment_{ENV}.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print(f'\nEnvironment:        {ENV}')
print(f'Ground truth:       [{TRUE_POSE[0]:+.3f}, {TRUE_POSE[1]:+.3f}, {TRUE_POSE[2]:+.3f}]')
print(f'Odometry guess:     [{ODOM_GUESS[0]:+.3f}, {ODOM_GUESS[1]:+.3f}, {ODOM_GUESS[2]:+.3f}]  (error: {np.linalg.norm(odom_error[:2]):.4f} m)')
print(f'Matched pose:       [{matched_pose[0]:+.3f}, {matched_pose[1]:+.3f}, {matched_pose[2]:+.3f}]  (error: {np.linalg.norm(match_error[:2]):.4f} m)')
print(f'Normalised score:   {score:.3f}')
print(f'Saved: scan_alignment_{ENV}.png')

## Cell 4 — Score Surface and Accuracy

Shows the score surface (dx vs dy at the best dtheta) and the covariance ellipse. The score surface reveals *why* some environments are harder to match — corridors produce an elongated ridge instead of a sharp peak.

Uses the same `ENV` variable from Cell 3. Change it and re-run both cells to compare environments.

In [ ]:
# Compute score surface (fix theta at matched value, sweep dx and dy)
ref_grid = matcher._build_local_grid(scan_ref)
sweep = 0.15
step = 0.005
x_vals = np.arange(matched_pose[0] - sweep, matched_pose[0] + sweep + step / 2, step)
y_vals = np.arange(matched_pose[1] - sweep, matched_pose[1] + sweep + step / 2, step)

print(f'Computing score surface ({len(x_vals)}x{len(y_vals)} = {len(x_vals)*len(y_vals)} evaluations)...')
scores_2d = np.zeros((len(y_vals), len(x_vals)))
for ix, x in enumerate(x_vals):
    for iy, y in enumerate(y_vals):
        scores_2d[iy, ix] = matcher._score_alignment(
            ref_grid, scan_new, np.array([x, y, matched_pose[2]]))
scores_2d /= max(len(scan_new), 1)
print('Done.')

# --- Plot ---
def plot_cov_ellipse(ax, centre, cov_2x2, n_std=2.0, **kwargs):
    eigvals, eigvecs = np.linalg.eigh(cov_2x2)
    eigvals = np.maximum(eigvals, 1e-10)
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    w = 2 * n_std * np.sqrt(eigvals[0])
    h = 2 * n_std * np.sqrt(eigvals[1])
    ax.add_patch(Ellipse(xy=centre, width=w, height=h, angle=angle, **kwargs))

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(scores_2d, extent=[x_vals[0], x_vals[-1], y_vals[0], y_vals[-1]],
               origin='lower', cmap='hot', aspect='equal')
ax.plot(TRUE_POSE[0], TRUE_POSE[1], 'g+', markersize=14, markeredgewidth=2, label='Ground truth')
ax.plot(matched_pose[0], matched_pose[1], 'c+', markersize=14, markeredgewidth=2, label='Matched pose')
ax.plot(ODOM_GUESS[0], ODOM_GUESS[1], 'wx', markersize=10, markeredgewidth=2, label='Odom guess')
plot_cov_ellipse(ax, (matched_pose[0], matched_pose[1]), covariance[:2, :2],
                 n_std=2.0, fill=False, edgecolor='cyan', linewidth=2, linestyle='--', label='2-sigma cov')
ax.set_xlabel('dx (m)'); ax.set_ylabel('dy (m)')
ax.set_title(f'Score Surface — {ENV}\nNormalised score at peak: {score:.3f}')
ax.legend(fontsize=9, loc='upper left')
plt.colorbar(im, ax=ax, label='Normalised score', shrink=0.8)
plt.tight_layout()
plt.savefig(f'score_surface_{ENV}.png', dpi=150, bbox_inches='tight')
plt.show()

# Print accuracy
print(f'\n--- Results for {ENV} ---')
print(f'Ground truth:       [{TRUE_POSE[0]:+.3f}, {TRUE_POSE[1]:+.3f}, {TRUE_POSE[2]:+.3f}]')
print(f'Matched pose:       [{matched_pose[0]:+.3f}, {matched_pose[1]:+.3f}, {matched_pose[2]:+.3f}]')
print(f'Translation error:  {np.linalg.norm(match_error[:2]):.4f} m')
print(f'Rotation error:     {np.degrees(abs(match_error[2])):.2f} deg')
print(f'Normalised score:   {score:.3f}')
print(f'Covariance diag:    sigma_x={np.sqrt(max(covariance[0,0],0)):.4f} m, '
      f'sigma_y={np.sqrt(max(covariance[1,1],0)):.4f} m, '
      f'sigma_theta={np.degrees(np.sqrt(max(covariance[2,2],0))):.2f} deg')
print(f'Saved: score_surface_{ENV}.png')